In [ ]:
import duckdb
import pandas as pd
import numpy as np
import os
import time

import pyrootutils
root = pyrootutils.setup_root(
    search_from=os.path.abspath(''),
    indicator=[".git"],
    pythonpath=True, # add root directory to the PYTHONPATH (helps with imports)
    dotenv=True, # load environment variables from .env if exists in root directory
)
from utils.extractor import Extractor
from src.utils.file_management.config_loader import load_yaml, process_config_values
from src.utils.file_management.file_manager import FileManager

import matplotlib.pyplot as plt

In [ ]:
datapath = 'PATH_TO_DATABASE' 
data_asset = 'DEID_CDW' # data_asset - Either DEID_CDW, DEID_OMOP, or PEDB

# Enter tables you are planning to query here
    
imagingfact             = duckdb.read_parquet(os.path.join(datapath,data_asset,'imagingfact','*.parquet'))
medicationorderfact     = duckdb.read_parquet(os.path.join(datapath,data_asset,'medicationorderfact','*.parquet'))
diagnosiseventfact      = duckdb.read_parquet(os.path.join(datapath,data_asset,'diagnosiseventfact','*.parquet'))
diagnosisterminologydim = duckdb.read_parquet(os.path.join(datapath,data_asset,'diagnosisterminologydim','*.parquet'))
procedureeventfact      = duckdb.read_parquet(os.path.join(datapath,data_asset,'procedureeventfact','*.parquet'))
procedureterminologydim = duckdb.read_parquet(os.path.join(datapath,data_asset,'procedureterminologydim','*.parquet'))
patdurabledim           = duckdb.read_parquet(os.path.join(datapath,data_asset,'patdurabledim','*.parquet'))
billingaccountfact      = duckdb.read_parquet(os.path.join(datapath,data_asset,'billingaccountfact','*.parquet'))
coveragedim             = duckdb.read_parquet(os.path.join(datapath,data_asset,'coveragedim','*.parquet'))

#Set print counts flag
check_query_flag = True       #axilluary checks to see if query makes sense
PlumsExtractor = Extractor(num_results_flag = True, display_results_flag = True)


In [ ]:
patient_list_path = 'project_patient_list.csv'

df = pd.read_csv(patient_list_path)
patientdurablekey_workinglist = df['patientdurablekey'].to_list()
print(len(patientdurablekey_workinglist))

# Extraction

### Preparation - reference table with dates of first LBP diagnosis

In [ ]:
# PATIENTS WITH LBP
diagnosisQuery = f'''
/*
Created By: Michelle Tong
Create Date: 9/29/2023
Description: Select patients with LBP
*/

SELECT DISTINCT 
  diagnosiskey, 
  diagnosisname, 
  type, 
  value
FROM diagnosisterminologydim
WHERE 
  (
    (
      LOWER(diagnosisname) LIKE '%back%pain%'
      AND 
      (
      LOWER(diagnosisname) LIKE '%lumb%'
      OR LOWER(diagnosisname) LIKE '%low%'
      )
    )
    OR LOWER(diagnosisname) LIKE '%lbp%'
    OR
    (
      value LIKE 'M51.1'
      OR value LIKE 'M54.4'
      OR value LIKE 'M54.9'
    )
  )
  --AND 
  --(
    --type LIKE 'ICD-10-CM'
    --OR type LIKE 'ICD-9-CM'
  --)
ORDER BY type, value
'''

# Run query and update relevant keys
results_df_pd = PlumsExtractor.run_query(diagnosisQuery, runtime_flag=True)

LBP_key_workinglist = results_df_pd['diagnosiskey'].drop_duplicates().tolist()
print('number of diagnosis keys:',len(LBP_key_workinglist))

results_df_pd[0:10]

In [ ]:
# DATE OF FIRST LBP
dates_query = f"""
/*
Created By: Michelle Tong
Create Date: 9/29/2023
Description: Select reference dates for LBP
*/

with tmp_diag_df AS (
  SELECT DISTINCT 
    patientdurablekey,
    encounterkey,
    diagnosisname,
    startdatekey,
    startdatekeyvalue
  FROM 
    diagnosiseventfact
  WHERE
    diagnosiskey IN {tuple(LBP_key_workinglist)}
    AND
    patientdurablekey IN {tuple(patientdurablekey_workinglist)}
  
  ORDER BY
    startdatekey,
    patientdurablekey
  )

SELECT DISTINCT
  d.patientdurablekey,
  d.startdatekey as startdatekey_firstLBP,
  d.startdatekeyvalue,
  p.birthdate,
  DATE_DIFF('year', p.birthdate, d.startdatekeyvalue) AS age_firstLBP
FROM tmp_diag_df as d
  JOIN patdurabledim as p
  ON 
    d.patientdurablekey = p.patientdurablekey
  WHERE birthdate IS NOT NULL
  ORDER BY 
    p.patientdurablekey ASC, age_firstLBP ASC
"""

df_dates = PlumsExtractor.run_query(dates_query, runtime_flag=True)
#Check whether query makes sense
if check_query_flag==True:
    PlumsExtractor.col_to_list(df_dates,'patientdurablekey')
    
df_dates = df_dates[~df_dates['age_firstLBP'].isin([np.nan, np.inf, -np.inf])] #.any(1)
df_dates = df_dates.loc[df_dates.groupby('patientdurablekey')['age_firstLBP'].idxmin()]
#Check whether query makes sense
if check_query_flag==True:
    PlumsExtractor.col_to_list(df_dates,'patientdurablekey')
    
df_dates = df_dates[df_dates['startdatekey_firstLBP']>20110101]
#Check whether query makes sense
if check_query_flag==True:
    PlumsExtractor.col_to_list(df_dates,'patientdurablekey')

#patients on 3/7/2025
#55,102 from 1990-2024
#50,682 from 2000-2024

# # SAVE OUTPUT
# save_path = 'project_patient_date_info.csv'
# os.makedirs(os.path.dirname(save_path), exist_ok=True) 
# df_dates.to_csv(save_path, index=False)

patientdurablekey_list = df_dates['patientdurablekey'].to_list()
print(len(patientdurablekey_list))

df_dates.head()

In [ ]:
# VISUALIZE DATES
# Extract year (first 4 digits)
start_year = df_dates['startdatekey_firstLBP'].astype(str).str[:4].astype(int)

# Plot histogram
#start_year.value_counts().sort_index().plot(kind='bar', figsize=(10, 6), rot=0)
plt.figure(figsize=(10, 6))
plt.hist(start_year, bins=range(2000, 2024 + 1), edgecolor='black', align='left')

plt.title('Count of Records by Year')
plt.xlabel('Year')
plt.ylabel('Count')
plt.show()

# Extract year (first 4 digits)
start_year = df_dates['age_firstLBP'].astype(int)

# Plot histogram
#start_year.value_counts().sort_index().plot(kind='bar', figsize=(10, 6), rot=0)
plt.figure(figsize=(10, 6))
plt.hist(start_year, edgecolor='black', align='left')

plt.title('Count of Records by Age')
plt.xlabel('Age')
plt.ylabel('Count')
plt.show()

### Outcome - Opioid within 6 months of first low back pain diagnosis

In [ ]:
# OPIOID PRESCRIPTIONS
#WHO guidelines - Analgesic https://www.ncbi.nlm.nih.gov/books/NBK554435/

#STEP 2 - Weak Opioids (less addictive)
opioid_step2_list = ['TRAMADOL', 'TILIDIN', 'TILIDINE', 'CODEINE','CODONE','HYDROCODONE']
opioid_step2_query_txt = ' OR '.join(['UPPER(medicationname) LIKE \'%'+x+'%\'' for x in opioid_step2_list])

#STEP 3  - Strong Opioids
opioid_step3_list = ['OXYCONTIN','VICODIN','OXYCODONE','HYDROCODONE','NORCO','FENTANYL','FENTANIL','FENTAN_L',
               'MORPHINE','MORPHONE','%MORPH_NE%','MEPERIDINE','BUPRENORPHINE','DILAUDID', 'METHADONE', 
                'BELBUCA', 'TAPENTADOL', 'LEVORPHANOL']
opioid_step3_query_txt = ' OR '.join(['UPPER(medicationname) LIKE \'%'+x+'%\'' for x in opioid_step3_list])

medsQuery = f'''
/*
Created By: Michelle Tong
Create Date: 6/11/2024
Description: Select relevant info (opioid vs nsaid meds) and keys from medication table for study cohort.
*/

SELECT DISTINCT 
  patientdurablekey,
  medicationorderkey, -- 
  encounterkey, -- 
  medicationkey, -- 
  medicationname, 
  ordername, -- 
  medicationtherapeuticclass,
  medicationpharmaceuticalclass, 
  medicationpharmaceuticalsubclass, -- 
  medicationstrength, -- 
  medicationform, -- 
  medicationroute, -- 
  route, -- 
  frequency, -- *
  doseunit, -- 
  durationkey, -- *
  startdatekey, -- *
  enddatekey, -- *
  orderedbyprovidertype,
  orderedbyproviderprimaryspecialty,
  authorizedbyprovidertype,
  authorizedbyproviderprimaryspecialty,
  class,
  mode,
  
  CASE
  WHEN {opioid_step3_query_txt} THEN 'STEP 3 OPIOID'
  WHEN {opioid_step2_query_txt} THEN 'STEP 2 OPIOID'
  ELSE 'Unspecified'
  END AS opioidtype
  
FROM 
  medicationorderfact
  WHERE patientdurablekey IN {tuple(patientdurablekey_list)}
    AND startdatekey > 0
    -- for perscribed medication
    AND UPPER(mode) LIKE 'OUTPATIENT'
    
    AND
    (
        --FOR OPIOIDS
        (
          (
          UPPER(medicationtherapeuticclass) LIKE 'ANALGESICS'
          AND UPPER(medicationpharmaceuticalsubclass) LIKE '%OPIOID%'
          AND UPPER(medicationpharmaceuticalsubclass) NOT LIKE '%NON-OPIOID%'
          AND UPPER(medicationpharmaceuticalsubclass) NOT LIKE '%DIETARY SUPPLEMENT%'
          AND 
          (
            UPPER(medicationpharmaceuticalsubclass) LIKE '% AGONISTS%' --not antagonists
            OR UPPER(medicationpharmaceuticalsubclass) LIKE '%CODEINE%'
            OR UPPER(medicationpharmaceuticalsubclass) LIKE '%FENTANYL%'
            OR UPPER(medicationpharmaceuticalsubclass) LIKE '%CODONE%'
            OR UPPER(medicationpharmaceuticalsubclass) LIKE '%HYDROMORPHONE%'
            OR UPPER(medicationpharmaceuticalsubclass) LIKE '%MEPERIDINE%'
            OR UPPER(medicationpharmaceuticalsubclass) LIKE '%PENTAZOCINE%'
            OR UPPER(medicationpharmaceuticalsubclass) LIKE '%PROPOXYPHENE%'
            OR UPPER(medicationpharmaceuticalsubclass) LIKE '%TRAMADOL%'
          )
          )
          -- in some cases the medicationpharamceutricalsubclass is '*Unspecified' and the 
          -- medicationname is an opioid. Exclude cases because the purpose is likely different
        )
    )

'''

# Run query and update relevant keys
df_meds_info = PlumsExtractor.run_query(medsQuery,runtime_flag=True,df_type='pandas')

#Check whether query makes sense
if check_query_flag==True:
    PlumsExtractor.col_to_list(df_meds_info, 'patientdurablekey')
    print('MedicationOrderFact -  total medication names: ')
    PlumsExtractor.col_to_list(df_meds_info, 'medicationname')

df_meds_info[0:10]

In [ ]:
# Intervention Labels
interventionQuery = f'''
/*
Description: Keep only the earliest startdatekey per patient. 
If multiple rows share the same date, drop rows with only 'other' labels if better options exist.
*/

WITH ranked_interventions AS (
    SELECT 
        a.patientdurablekey,
        a.startdatekey,
        --a.medicationname,
        --a.medicationstrength,
        CASE 
            WHEN a.medicationpharmaceuticalsubclass LIKE 'Analgesic Opioid Agonists ' THEN 'Analgesic Opioid Agonists '
            WHEN a.medicationpharmaceuticalsubclass LIKE 'Analgesic Opioid %codone Combinations' THEN 'Analgesic Opioid Hydrocodone/Oxycodone Combinations'
            ELSE 'other'
        END AS medicationpharmaceuticalsubclass,
        CASE 
            WHEN a.orderedbyprovidertype LIKE 'Physician' THEN 'Physician'
            WHEN a.orderedbyprovidertype LIKE '%Nurse%' THEN 'Nurse'
            WHEN a.orderedbyprovidertype LIKE '%Resident%' THEN 'Resident'
            WHEN a.orderedbyprovidertype LIKE '%Physician Assistant%' THEN 'Physician Assistant'
            ELSE 'other'
        END AS orderedbyprovidertype,
        CASE 
            WHEN a.orderedbyproviderprimaryspecialty LIKE 'Orthopedic Surgery' THEN 'Orthopedic Surgery'
            WHEN a.orderedbyproviderprimaryspecialty LIKE 'Family Medicine' THEN 'Family Medicine'
            WHEN a.orderedbyproviderprimaryspecialty LIKE 'Internal Medicine' THEN 'Internal Medicine'
            WHEN a.orderedbyproviderprimaryspecialty LIKE 'Pain Medicine' THEN 'Pain Medicine'
            ELSE 'other'
        END AS orderedbyproviderprimaryspecialty,
        ROW_NUMBER() OVER (
            PARTITION BY a.patientdurablekey 
            ORDER BY a.startdatekey ASC,
                     -- prioritize more specific rows over 'other'
                     CASE WHEN a.medicationpharmaceuticalsubclass LIKE 'other' THEN 0 ELSE 1 END,
                     CASE WHEN a.orderedbyprovidertype LIKE 'other' THEN 0 ELSE 1 END,
                     CASE WHEN a.orderedbyproviderprimaryspecialty LIKE 'other' THEN 0 ELSE 1 END
            ) AS row_rank
    FROM 
        df_meds_info AS a
    LEFT JOIN 
        df_dates AS b
    ON 
        a.patientdurablekey = b.patientdurablekey
    WHERE 
        a.patientdurablekey IN {tuple(patientdurablekey_list)}
        AND a.startdatekey BETWEEN b.startdatekey_firstLBP AND b.startdatekey_firstLBP + 600
)

SELECT 
    patientdurablekey,
    startdatekey,
    medicationpharmaceuticalsubclass,
    orderedbyprovidertype,
    orderedbyproviderprimaryspecialty
FROM 
    ranked_interventions
WHERE 
    row_rank = 1
ORDER BY 
    patientdurablekey

'''

# Run query and update relevant keys
df_meds = PlumsExtractor.run_query(interventionQuery,runtime_flag=True,df_type='pandas')
df_meds['opioid'] =  1

# # SAVE OUTPUT
# save_path = 'medicationorderfact.csv'
# os.makedirs(os.path.dirname(save_path), exist_ok=True) 
# df_dates.to_csv(save_path, index=False)

#Check whether query makes sense
if check_query_flag==True:
    PlumsExtractor.col_to_list(df_meds, 'patientdurablekey')
#    print('MedicationOrderFact -  total medication names: ')
#    PlumsExtractor.col_to_list(results_df_pd, 'interventiontype')
    
df_meds.head()

### Predictor - Demographics

In [ ]:
# Demographics
demographicsQuery = f'''
/*
Created By: Michelle Tong
Create Date: 6/10/2024
Description: Select relevant demographics (sex, ethnicity, birthday, postal code, ...) for study cohort
*/

SELECT a.*,
  b.age_firstLBP,
  b.startdatekey_firstLBP
FROM
  (SELECT DISTINCT
    patientdurablekey, 
    patientepicid, 
    sex, 
    preferredlanguage, 
    ucsfderivedraceethnicity_x, 
    birthdate, 
    deathdate,
    --address, DEID
    --city, DEID
    stateorprovince,
    --county, DEID
    addresskey,
    postalcode, 
    maritalstatus,
    religion,
    smokingstatus,
    primaryfinancialclass, 
    -- highestlevelofeducation, *Unspecified
    sexassignedatbirth, 
    genderidentity 
  FROM 
    patdurabledim as a
    WHERE 
      patientdurablekey IN {tuple(patientdurablekey_list)}
  ) as a
LEFT JOIN df_dates as b 
ON a.patientdurablekey = b.patientdurablekey
'''

# Run query and update relevant keys
results_df_pd = PlumsExtractor.run_query(demographicsQuery,runtime_flag=True,df_type='pandas')

# # SAVE OUTPUT
# save_path = 'patdurabledim.csv'
# os.makedirs(os.path.dirname(save_path), exist_ok=True) 
# df_dates.to_csv(save_path, index=False)

#Check whether query makes sense
if check_query_flag==True:
    PlumsExtractor.col_to_list(results_df_pd,'patientdurablekey')

results_df_pd[0:10]

In [ ]:
# Intervention Labels
dataQuery = f'''
/*
Created By: Michelle Tong
Create Date: 
Description: For analysis
*/

SELECT DISTINCT
    patientdurablekey,
    sex,
    age_firstLBP,
    startdatekey_firstLBP, --CAST(SUBSTR(CAST(startdatekey AS VARCHAR), 1, 4) AS INTEGER) as yearatLBP,
    CASE WHEN preferredlanguage LIKE 'English' THEN 'english'
        ELSE 'Non-english'
        END AS preferredlanguage,
    CASE WHEN ucsfderivedraceethnicity_x LIKE 'White' THEN 'white'
        WHEN ucsfderivedraceethnicity_x LIKE 'Asian' THEN 'asian'
        WHEN ucsfderivedraceethnicity_x LIKE 'Latinx' THEN 'latinx'
        WHEN ucsfderivedraceethnicity_x LIKE 'Black or African American' THEN 'black or african american'
        WHEN ucsfderivedraceethnicity_x LIKE 'Unknown/Declined' THEN 'unknown'
        ELSE 'other' -- captures other, multi-race, southwest asian, north african, pacific islander, native american
        END AS raceethnicity,
    CASE WHEN smokingstatus LIKE 'Every Day' THEN 'smoker'
        WHEN smokingstatus LIKE 'Heavy Smoker' THEN 'smoker'
        WHEN smokingstatus LIKE 'Light Smoker' THEN 'smoker'
        WHEN smokingstatus LIKE 'Some Days' THEN 'smoker'
        WHEN smokingstatus LIKE 'Smoker, Current Status Unknown' THEN 'smoker'
        WHEN smokingstatus LIKE 'Former' THEN 'former'
        WHEN smokingstatus LIKE 'Passive Smoke Exposure - Never Smoker' THEN 'never'
        WHEN smokingstatus LIKE 'Never' THEN 'never'
        ELSE 'unknown'
        END AS smokingstatus,
    CASE WHEN religion LIKE 'Unknown' THEN 'unknown'
        WHEN religion LIKE '*Unspecified' THEN 'unknown'
        WHEN religion LIKE 'none' THEN 'none'
        WHEN religion LIKE 'No Faith' THEN 'none'
        WHEN religion LIKE 'Non-Denominational' THEN 'none'
        WHEN religion LIKE 'Spiritual & Not Religious' THEN 'none'
        ELSE 'religious' 
        END AS religion,
    CASE WHEN maritalstatus LIKE 'Married' THEN 'partnered'
        WHEN maritalstatus LIKE 'Significant Other' THEN 'partnered'
        WHEN maritalstatus LIKE 'Registered Domestic Partner' THEN 'partnered'
        WHEN maritalstatus LIKE '' THEN 'partnered'
        WHEN maritalstatus LIKE 'Single' THEN 'single/separated'
        WHEN maritalstatus LIKE 'Divorced' THEN 'single/separated'
        WHEN maritalstatus LIKE 'Widowed' THEN 'single/separated'
        WHEN maritalstatus LIKE 'Legally Separated' THEN 'single/separated'
        WHEN maritalstatus LIKE 'RDP-Widowed' THEN 'single/separated' 
            -- can use Reg Dom Partner filling status for 2 years after change in status for benefits 
        WHEN maritalstatus LIKE 'RDP-Dissolved' THEN 'single/separated' 
            -- can use Reg Dom Partner filling status for 2 years after change in status for benefits 
        ELSE 'unknown'
        END AS socialsupport

FROM 
  results_df_pd
WHERE 
    LOWER(sex) IN {('female','male')} 

ORDER BY
  patientdurablekey
'''

# Run query and update relevant keys
df_demographics = PlumsExtractor.run_query(dataQuery,runtime_flag=True,df_type='pandas')

#Check whether query makes sense
if check_query_flag==True:
    PlumsExtractor.col_to_list(df_demographics, 'patientdurablekey')

# # SAVE OUTPUT
# save_path = 'patdurabledim_analysis.csv'
# os.makedirs(os.path.dirname(save_path), exist_ok=True) 
# df_dates.to_csv(save_path, index=False)

# All subjects may not have had the opportunity to specify religion which would introduce unintended biases
df_demographics = df_demographics.drop('religion',axis=1) 

df_demographics.head()

### Predictor - Billing

In [ ]:
# Select relevant patient diagnoses
financeQuery = f'''
/*
Created By: Michelle Tong
Created Date: 8/30/2024
Description: Select financial info for patient cohort.
*/

SELECT 
  b.patientdurablekey,
  b.encounterkey,
  b.primaryencounterkey,
  b.departmentname,
  b.departmentspecialty,
  b.financialclass,
  b.billstatus,
  cdim1.coveragefinancialclass AS fc1_cov,
  cdim1.payorfinancialclass AS fc1_pay,
  cdim1.benefitplanproducttype AS fc1_plan,
  cdim2.coveragefinancialclass AS fc2_cov,
  cdim2.payorfinancialclass AS fc2_pay,
  cdim2.benefitplanproducttype AS fc2_plan
FROM billingaccountfact as b
LEFT JOIN coveragedim cdim1
    ON cdim1.coveragekey=b.primarycoveragekey
LEFT JOIN coveragedim cdim2 ON
    cdim2.coveragekey=b.secondcoveragekey
WHERE
  b.patientdurablekey IN {tuple(patientdurablekey_list)}

'''

# Note - very few results when joining on encounter key

# Run query and update relevant keys
results_df_pd = PlumsExtractor.run_query(financeQuery,runtime_flag=True,df_type='pandas')

# # SAVE OUTPUT
# save_path = 'billingaccountfact.csv'
# os.makedirs(os.path.dirname(save_path), exist_ok=True) 
# df_dates.to_csv(save_path, index=False)

#Check whether query makes sense
if check_query_flag==True:
    PlumsExtractor.col_to_list(results_df_pd, 'fc1_plan')
    
results_df_pd[0:10]

In [ ]:
# Intervention Labels
dataQuery = f'''
/*
Created By: Michelle Tong
Create Date: 
Description: For analysis
*/

  SELECT DISTINCT
    patientdurablekey,
    encounterkey,
    CASE WHEN fc1_plan LIKE 'PPO' THEN 'PPO' -- preferred provider organizations
        WHEN fc1_plan LIKE '%Medicare%' THEN 'medicare' -- older using social security
        WHEN fc1_plan LIKE 'Managed Medi-Cal' THEN 'medicaid' -- low-income
        WHEN fc1_plan LIKE 'Medi-Cal Standard' THEN 'medicaid' -- low-income
        WHEN fc1_plan LIKE 'Medicaid Out-of-State ' THEN 'medicaid' -- low-income
        WHEN fc1_plan LIKE 'HMO' THEN 'HMO/POS/EPO'
        WHEN fc1_plan LIKE 'IPA' THEN 'HMO/POS/EPO'
        WHEN fc1_plan LIKE 'POS' THEN 'HMO/POS/EPO' -- between PPO and HMO, closer to HMO
        WHEN fc1_plan LIKE 'EPO' THEN 'HMO/POS/EPO' -- between PPO and HMO, closer to HMO
        ELSE 'other' -- Worker's Comp, TriCare, GOV, Indemnity(sued), MIA
        END AS primaryinsurance,
    CASE WHEN fc1_plan LIKE 'PPO' THEN 5 -- preferred provider organizations
        WHEN fc1_plan LIKE '%Medicare%' THEN 4 -- older using social security
        WHEN fc1_plan LIKE 'Managed Medi-Cal' THEN 3 -- low-income
        WHEN fc1_plan LIKE 'Medi-Cal Standard' THEN 3 -- low-income
        WHEN fc1_plan LIKE 'Medicaid Out-of-State ' THEN 3 -- low-income
        WHEN fc1_plan LIKE 'HMO' THEN 2
        WHEN fc1_plan LIKE 'IPA' THEN 2
        WHEN fc1_plan LIKE 'POS' THEN 2 -- between PPO and HMO, closer to HMO
        WHEN fc1_plan LIKE 'EPO' THEN 2 -- between PPO and HMO, closer to HMO
        ELSE 1 -- Worker's Comp, TriCare, GOV, Indemnity(sued), MIA
        END AS primaryinsurancekey,
  FROM 
    results_df_pd
  WHERE 
    patientdurablekey IN {tuple(patientdurablekey_list)}
  ORDER BY
    patientdurablekey
'''

# Run query and update relevant keys
results_df_pd = PlumsExtractor.run_query(dataQuery,runtime_flag=True,df_type='pandas')

#Check whether query makes sense
if check_query_flag==True:
    PlumsExtractor.col_to_list(results_df_pd, 'patientdurablekey')

# # SAVE OUTPUT
# save_path = 'billingaccountfact_analysis.csv'
# os.makedirs(os.path.dirname(save_path), exist_ok=True) 
# df_dates.to_csv(save_path, index=False)

# Select row with the highest primaryinsurancekey for each patient
df_insurance = results_df_pd.loc[results_df_pd.groupby('patientdurablekey')['primaryinsurancekey'].idxmax()]

# Reset index for cleanliness
df_insurance = df_insurance.reset_index(drop=True).drop('encounterkey',axis=1)
#Check whether query makes sense
if check_query_flag==True:
    PlumsExtractor.col_to_list(df_insurance, 'patientdurablekey')

df_insurance.head()

In [ ]:
df_insurance.value_counts('primaryinsurance')

### Predictor - Diagnoses

In [ ]:
# Select relevant patient diagnoses

#infection in bone, intervertebral disc, vertebral endplate and vertebral body, disc and adjacent vertebral body, disc and adjacent vertebral body
exclude_diag_list = ['infection','osteomyelitis','discitis','spondylitis','spondylodiscitis','septic facet joint']
exclude_diag_query_txt = ' OR '.join(['LOWER(a.diagnosisname) LIKE \'%'+x+'%\'' for x in exclude_diag_list])

# 
#   CASE
#       WHEN LOWER(a.diagnosisname) LIKE '%back%pain%' THEN 'low back pain'
#       WHEN LOWER(a.diagnosisname) LIKE '%radicul%' THEN 'radiculopathy'
#       WHEN LOWER(a.diagnosisname) LIKE '%spinal stenosis%' THEN 'spinal stenosis'
#       WHEN (LOWER(a.diagnosisname) LIKE '%herniated%disc%' OR LOWER(a.diagnosisname) LIKE '%disc%herniation%') THEN 'herniated disc'
#       WHEN LOWER(a.diagnosisname) LIKE '%fusion%' THEN 'fusion'
#       WHEN LOWER(a.diagnosisname) LIKE '%fracture%' THEN 'fracture'
#       WHEN LOWER(a.diagnosisname) LIKE '%failed back%' THEN 'failed back'
#       WHEN LOWER(a.diagnosisname) LIKE '%scoliosis%' THEN 'scoliosis'
#       WHEN LOWER(a.diagnosisname) LIKE '%abscess%' THEN 'abscess'
#       WHEN ({exclude_diag_query_txt}) THEN 'infection'
#       WHEN (LOWER(a.diagnosisname) LIKE '%tumor%' OR LOWER(a.diagnosisname) LIKE '%metastasis%' OR LOWER(a.diagnosisname) LIKE '%oncology%') THEN 'cancer'
#       WHEN LOWER(a.diagnosisname) LIKE '%depression%' THEN 'depression'
#       WHEN LOWER(a.diagnosisname) LIKE '%anxiety%' THEN 'anxiety'
#       ELSE 'category not specified'
#       END AS diagnosistype
        
diagnosisQuery = f'''
/*
Created By: Michelle Tong
Description: Select relevant diagnoses for patient cohort.
*/
SELECT DISTINCT
  a.patientdurablekey,
  a.diagnosiskey,
  a.diagnosisname,
  a.startdatekey,
  a.departmentname, 
  a.departmentspecialty,
  a.encounterkey, -- 
  a.departmentkey, -- 
  a.enddatekey, -- 
  a.documentedbyprovidertype, -- 
  a.documentedbyproviderprimaryspecialty, -- 
  a.type
FROM (
  SELECT 
    * 
  FROM
    diagnosiseventfact
    WHERE
      patientdurablekey IN {tuple(patientdurablekey_list)}
      AND startdatekey > 0
) AS a

LEFT JOIN df_dates AS b
ON a.patientdurablekey = b.patientdurablekey
WHERE 
    a.patientdurablekey IN {tuple(patientdurablekey_list)}
    AND a.startdatekey < b.startdatekey_firstLBP
ORDER BY 
    a.patientdurablekey
'''

# Run query and update relevant keys
results_df_pd = PlumsExtractor.run_query(diagnosisQuery,runtime_flag=True,df_type='pandas')

# # SAVE OUTPUT
# save_path = 'diagnosiseventfact.csv'
# os.makedirs(os.path.dirname(save_path), exist_ok=True) 
# df_dates.to_csv(save_path, index=False)

#Check whether query makes sense
if check_query_flag==True:
    PlumsExtractor.col_to_list(results_df_pd, 'patientdurablekey')
    PlumsExtractor.col_to_list(results_df_pd, 'diagnosisname')
    
results_df_pd[0:10]

In [ ]:
# Intervention Labels
inputQuery = f'''
/*
Created By: Michelle Tong
Create Date: 
Description: For analysis
*/

SELECT DISTINCT
  a.patientdurablekey,
  --diagnosisname,
  
  -- START DEMOGRAPHICS/PSYCHOSOCIAL
  CASE WHEN LOWER(a.diagnosisname) LIKE '%anxiety%' THEN 1
      ELSE 0
      END AS 'anxiety',
  CASE WHEN LOWER(a.diagnosisname) LIKE '%depression%' THEN 1
      ELSE 0
      END AS 'depression',
  CASE WHEN (Anxiety==1 OR Depression==1) THEN 1
    ELSE 0
    END AS 'negativepsychstate',
  CASE WHEN (LOWER(a.diagnosisname) LIKE '%obes%' AND LOWER(a.diagnosisname) NOT LIKE '%nonobes%') THEN 1
      ELSE 0
      END AS 'obesity',
  -- END DEMOGRAPHICS/PSYCHOSOCIAL
  
  -- START SYMPTOMS/DIAGNOSES
  CASE WHEN (LOWER(a.diagnosisname) LIKE '%back%pain%' AND LOWER(a.diagnosisname) LIKE '%acute%') THEN 1 -- 'acute'
      WHEN (LOWER(a.diagnosisname) LIKE '%back%pain%') AND (LOWER(a.diagnosisname) LIKE '%chronic%') 
          AND (LOWER(a.diagnosisname) NOT LIKE '%unspecif%chronic%') THEN 2 -- 'chronic'
      ELSE 0 -- 'unspecified'
      END AS 'lbpduration',
  CASE WHEN (LOWER(a.diagnosisname) LIKE '%back%pain%' AND LOWER(a.diagnosisname) LIKE '%right%') THEN 1 -- 'right-sided'
      WHEN ((LOWER(a.diagnosisname) LIKE '%back%pain%') AND (LOWER(a.diagnosisname) LIKE '%left%')) THEN 2 -- 'left-sided' 
      WHEN ((LOWER(a.diagnosisname) LIKE '%back%pain%') AND (LOWER(a.diagnosisname) LIKE '%bilateral%')) THEN 3 --'bilateral' 
      ELSE 0 --'unspecified'
      END AS 'lbplaterality',
  CASE WHEN (LOWER(diagnosisname) LIKE '%radiculopathy%' OR
     (LOWER(diagnosisname) LIKE '%radiat%' AND 
     (LOWER(diagnosisname) LIKE '%back%pain%'OR LOWER(diagnosisname) LIKE '%lbp%'))) THEN 1 -- radiating, radiation
      ELSE 0
      END AS 'radiculopathy',
  CASE WHEN (LOWER(diagnosisname) LIKE '%numbness%' OR LOWER(diagnosisname) LIKE '%tingling%') THEN 1 
      ELSE 0
      END AS 'numbnesstingling',
  CASE WHEN (LOWER(a.diagnosisname) LIKE '%sciatic%' AND LOWER(a.diagnosisname) NOT LIKE '%without%') THEN 1
      ELSE 0
      END AS 'sciatica',
  CASE WHEN LOWER(a.diagnosisname) LIKE '%listhesis%'  THEN 1 --spondylo, antero, retro
      ELSE 0
      END AS 'listhesis',
  CASE WHEN ((LOWER(a.diagnosisname) LIKE '%facet%' OR LOWER(diagnosisname) LIKE '%joint%')
          AND LOWER(a.diagnosisname) LIKE '%arthropathy%') THEN 1
      ELSE 0
      END AS 'facetjointarthropathy',
  CASE WHEN LOWER(a.diagnosisname) LIKE '%scoliosis%' THEN 1
      ELSE 0
      END AS 'scoliosis',
  CASE WHEN ((LOWER(a.diagnosisname) LIKE '%disc %' 
      OR LOWER(a.diagnosisname) LIKE '%disc'
      OR LOWER(a.diagnosisname) LIKE '%disc,%'
      OR LOWER(a.diagnosisname) LIKE '%discogenic%'
      OR LOWER(a.diagnosisname) LIKE '%discectomy%'
      ) AND LOWER(a.diagnosisname) NOT LIKE '%optic%') THEN 1 --discomfort, discharge, discussion, discrepancy)
      ELSE 0
      END AS 'discpathology',
  CASE WHEN (LOWER(a.diagnosisname) LIKE '%stenosis%' AND (LOWER(a.diagnosisname) LIKE '%spin%' 
          OR LOWER(a.diagnosisname) LIKE '%lumb%' OR LOWER(a.diagnosisname) LIKE '%foramina%'
          OR LOWER(a.diagnosisname) LIKE '%thoracic%')) THEN 1
      ELSE 0
      END AS 'spinalstenosis',
  CASE WHEN (LOWER(a.diagnosisname) LIKE '%sacroiliac%' 
      OR LOWER(a.diagnosisname) LIKE '% si %'
      OR LOWER(a.diagnosisname) LIKE 'si %') THEN 1
      ELSE 0
      END AS 'sacroiliacjoint',
  CASE WHEN (LOWER(a.diagnosisname) LIKE '%osteoarthritis%'
      OR LOWER(a.diagnosisname) LIKE '%osteoarthrosis%') THEN 1
      ELSE 0
      END AS 'osteoarthritisososteoarthritis',
  CASE WHEN (LOWER(a.diagnosisname) LIKE '%osteoporosis%'
      OR LOWER(a.diagnosisname) LIKE '%osteopenia%') THEN 1
      ELSE 0
      END AS 'osteopeniaosteoporosis',
  CASE WHEN (LOWER(diagnosisname) LIKE '%bladder%' OR 
      (LOWER(diagnosisname) LIKE '%bladder%' 
      AND (LOWER(diagnosisname) LIKE '%dysfunction%'
      OR LOWER(diagnosisname) LIKE '%disorder%'
      OR LOWER(diagnosisname) LIKE '%irrita%')) ) THEN 1 -- iritable, irritation
      ELSE 0
      END AS 'bowelbladder',
  CASE WHEN (LOWER(a.diagnosisname) LIKE '%diabet%' AND LOWER(a.diagnosisname) NOT LIKE '%history%'
       AND LOWER(a.diagnosisname) NOT LIKE '%screening%' AND LOWER(a.diagnosisname) NOT LIKE '%antidiabet%' 
       AND LOWER(a.diagnosisname) NOT LIKE '%pre-diabet%') THEN 1 -- diabetes, diabetic
       ELSE 0
       END AS 'diabetes'
  -- END SYMPTOMS/DIAGNOSES
FROM 
  results_df_pd as a

ORDER BY
  a.patientdurablekey
'''

# Run query and update relevant keys
df_diagnoses = PlumsExtractor.run_query(inputQuery,runtime_flag=True,df_type='pandas')

# Modify labels
df_diagnoses = df_diagnoses.groupby('patientdurablekey', as_index=True).max().reset_index()

#Check whether query makes sense
if check_query_flag==True:
    PlumsExtractor.col_to_list(df_diagnoses, 'patientdurablekey')
    
# # SAVE OUTPUT
# save_path = 'diagnosiseventfact_analysis.csv'
# os.makedirs(os.path.dirname(save_path), exist_ok=True) 
# df_dates.to_csv(save_path, index=False)

df_diagnoses.head()

In [ ]:
df_diagnoses.describe()


### Predictors - Imaging

In [ ]:
# Imaging
imagingQuery = f'''
        /*
        Created By: Michelle Tong
        Create Date: 3/3/2023
        Description: Select relevant patient, visit, and image keys from imagingfact table for study cohort.
        */
        
        WITH imaging_data AS (
        
        SELECT DISTINCT
            accessionnumber, 
            patientdurablekey,
            encounterkey, 
            firstprocedurename, 
            examstartdatekey,
            examstartdatekeyvalue,
            orderingdepartmentname,
            orderingdepartmentspecialty,
            performingdepartmentname,
            performingdepartmentspecialty,
            CASE WHEN UPPER(firstprocedurename) LIKE '%MR%' THEN 'MR'
              WHEN UPPER(firstprocedurename) LIKE '%XR%' THEN 'XR'
              WHEN UPPER(firstprocedurename) LIKE '%CT%' THEN 'CT'
              END AS imagetype

        FROM 
            imagingfact

          WHERE patientdurablekey IN {tuple(patientdurablekey_list)}
          AND examstartdatekey > 0
          AND canceled=0
          
          AND 
            (
              UPPER(firstprocedurename) LIKE '%LUMB%' -- LUMBAR
              OR UPPER(firstprocedurename) LIKE '%L_SPINE%' -- L-SPINE & L SPINE, FETAL SPINE, TOTAL SPINE
            )
            AND 
            (
              UPPER(firstprocedurename) LIKE '%MR%'
              OR UPPER(firstprocedurename) LIKE '%XR%'
              OR UPPER(firstprocedurename) LIKE '%CT%'
            )

            AND UPPER(firstprocedurename) NOT LIKE '%CERV%' -- CERVICAL
            AND UPPER(firstprocedurename) NOT LIKE '%THOR%' -- THORACIC
            AND UPPER(firstprocedurename) NOT LIKE '%FETAL%'
            AND UPPER(firstprocedurename) NOT LIKE '%TOTAL%'
            AND UPPER(firstprocedurename) NOT LIKE '%OUTSIDE%' -- exam at occured outside UCSF
            
          ORDER BY
            patientdurablekey,
            examstartdatekey   
        )
        SELECT 
            a.*,
            -- Convert BIGINT YYYYMMDD to an integer representation of days difference
            CAST(
            SUBSTR(CAST(b.startdatekey_firstLBP AS STRING), 1, 4) || '-' || 
            SUBSTR(CAST(b.startdatekey_firstLBP AS STRING), 5, 2) || '-' || 
            SUBSTR(CAST(b.startdatekey_firstLBP AS STRING), 7, 2) 
            AS TIMESTAMP
            ) AS start_date_formatted,
            
            CAST(
            SUBSTR(CAST(a.examstartdatekey AS STRING), 1, 4) || '-' || 
            SUBSTR(CAST(a.examstartdatekey AS STRING), 5, 2) || '-' || 
            SUBSTR(CAST(a.examstartdatekey AS STRING), 7, 2) 
            AS TIMESTAMP
            ) AS exam_date_formatted,

            -- Approximate difference in days (better to do this in Python after query execution)
            (start_date_formatted - exam_date_formatted) AS dayspriortoLBP

        FROM imaging_data AS a
        LEFT JOIN df_dates AS b
        ON a.patientdurablekey = b.patientdurablekey
        WHERE 
            a.patientdurablekey IN {tuple(patientdurablekey_list)}
            AND a.examstartdatekey BETWEEN b.startdatekey_firstLBP - 300 AND b.startdatekey_firstLBP +300 -- 3 months before or after LBP diagnosis
        ORDER BY 
            a.patientdurablekey,
            a.examstartdatekey
        '''

# Run query and update relevant keys
results_df_pd = PlumsExtractor.run_query(imagingQuery,runtime_flag=True,df_type='pandas')

# # SAVE OUTPUT
# save_path = 'imagingfact.csv'
# os.makedirs(os.path.dirname(save_path), exist_ok=True) 
# df_dates.to_csv(save_path, index=False)

#Check whether query makes sense
if check_query_flag==True:
    PlumsExtractor.col_to_list(results_df_pd,'patientdurablekey')

# Create binary indicators
df_imaging = pd.get_dummies(results_df_pd, columns=['imagetype'], prefix='', prefix_sep='')

# Group by patientdurablekey and take the max (to convert multiple rows to one)
df_imaging = df_imaging.groupby('patientdurablekey', as_index=False).max()

# Optional: reorder or rename columns if needed
df_imaging = df_imaging[['patientdurablekey', 'MR', 'XR', 'CT']]

df_imaging[0:10]

# Merge dataframes

In [ ]:
# Patients MUST have demographic and diagnosis information
valid_pdk = list(set(df_demographics['patientdurablekey']) & set(df_diagnoses['patientdurablekey']))

# Left join on patientdurablekey
print(len(df_meds), len(set(df_meds['patientdurablekey'])))
print(len(df_demographics), len(set(df_demographics['patientdurablekey'])))
print(len(df_insurance), len(set(df_insurance['patientdurablekey'])))
print(len(df_diagnoses), len(set(df_diagnoses['patientdurablekey'])))
print(len(df_imaging), len(set(df_imaging['patientdurablekey'])))

merged_df = df_demographics.merge(df_meds, on='patientdurablekey', how='left')
merged_df = merged_df.merge(df_insurance, on='patientdurablekey', how='left')
merged_df = merged_df.merge(df_diagnoses, on='patientdurablekey', how='left')
merged_df = merged_df.merge(df_imaging, on='patientdurablekey', how='left')
merged_df = merged_df[merged_df['patientdurablekey'].isin(valid_pdk)]
merged_df[['startdatekey','medicationpharmaceuticalsubclass','orderedbyprovidertype','orderedbyproviderprimaryspecialty','MR','XR','CT']].fillna(0, inplace = True) 
merged_df = merged_df.reset_index(drop=True)
print(len(merged_df))
print(merged_df.columns)

# SAVE OUTPUT
save_path = 'patient_profile_analysis.csv'
os.makedirs(os.path.dirname(save_path), exist_ok=True) 
df_dates.to_csv(save_path, index=False)

merged_df